# EXP-000 — Baseline: Faithful Reproduction of the Production Pipeline

**Purpose.** External anchor for the Phase 8 experiment series. This notebook reproduces the
*current production model* of the `fraud-detection-ml` repository — numeric-only features,
`fillna(0)`, sklearn `GradientBoostingClassifier` — scores the competition test set, and produces
`submission.csv`. Nothing is improved here on purpose: the resulting public/private LB score is
the "before" number every later experiment is measured against.

**Protocol.** `docs/kaggle/validation-protocol.md` v1 (Scheme A: temporal 80/20 holdout,
identical to production). Scheme B (GroupKFold) is exempt for EXP-000 — H4's scope is
EXP-001..004 and sklearn GB is single-threaded (6 extra fits would be prohibitive).

**Registry discipline.** EXP-000 registered in `docs/kaggle/experiment-registry.md` before this
run; submission logged as SUB-001 in `docs/kaggle/submission-log.md` before upload.

**Expected.** Holdout ROC-AUC ≈ 0.861 (reproduction check, ±0.003). Private LB expected below
the holdout (temporal drift; the test period starts after a one-month gap): rough guess 0.83–0.86.

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# --- Configuration: hardcoded copy of configs/model_gb_v1.yml for self-containment ---
SPLIT_QUANTILE = 0.8
GB_PARAMS = dict(
    n_estimators=80,
    max_depth=5,
    learning_rate=0.1,
    min_samples_leaf=100,
    subsample=0.8,
    random_state=42,
)
EXCLUDE_COLS = {"isFraud", "TransactionID", "TransactionDT"}

KAGGLE_INPUT = Path("/kaggle/input/ieee-fraud-detection")
ON_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

if ON_KAGGLE:
    if not KAGGLE_INPUT.exists():
        mounted = sorted(p.name for p in Path("/kaggle/input").glob("*"))
        raise FileNotFoundError(
            "Competition data not attached to this kernel. "
            "Fix in the notebook editor: Add Input -> Competitions -> "
            f"IEEE-CIS Fraud Detection. Currently mounted: {mounted}"
        )
    DATA_DIR = KAGGLE_INPUT
else:
    DATA_DIR = Path("../../../data/raw")  # local dry-run only
print(f"Data dir: {DATA_DIR}")

## 1. Load and merge training data

Mirrors `src/data/loader.py::load_full_training_dataset`: transaction LEFT JOIN identity on
`TransactionID`.

In [ ]:
train_transaction = pd.read_csv(DATA_DIR / "train_transaction.csv")
train_identity = pd.read_csv(DATA_DIR / "train_identity.csv")
df = train_transaction.merge(train_identity, on="TransactionID", how="left")
del train_transaction, train_identity

print(f"Shape: {df.shape}")
print(f"Fraud rate: {df['isFraud'].mean():.4f}")

## 2. Temporal split (Scheme A)

Mirrors `src/data/split.py::temporal_train_val_split`: cutoff at the 0.8 quantile of
`TransactionDT`; train strictly before, holdout at/after.

In [ ]:
cutoff = df["TransactionDT"].quantile(SPLIT_QUANTILE)
train_mask = df["TransactionDT"] < cutoff

y_train = df.loc[train_mask, "isFraud"].astype(int)
y_val = df.loc[~train_mask, "isFraud"].astype(int)
print(f"Train: {train_mask.sum():,} rows | Holdout: {(~train_mask).sum():,} rows")

## 3. Feature matrix

Mirrors `src/features/feature_registry.py::infer_numeric_feature_list` +
`src/features/pipeline.py::build_features`: numeric dtypes only, IDs/target/time excluded,
`fillna(0)`. The feature list is inferred from the TRAIN partition and reused everywhere —
same contract the API enforces from model metadata.

In [ ]:
feature_list = [
    c for c in df.columns if df[c].dtype != "O" and c not in EXCLUDE_COLS
]
print(f"Features: {len(feature_list)}")

X_train = df.loc[train_mask, feature_list].fillna(0.0)
X_val = df.loc[~train_mask, feature_list].fillna(0.0)
val_ids = df.loc[~train_mask, "TransactionID"].to_numpy()
del df

## 4. Train and validate

sklearn `GradientBoostingClassifier` is single-threaded — expect a long fit (roughly 1–3 h on
Kaggle CPU for ~470k × 380). This cost is part of what ADR-007 replaces from EXP-001 onward.

In [ ]:
t0 = time.time()
model = GradientBoostingClassifier(**GB_PARAMS)
model.fit(X_train, y_train)
print(f"Fit time: {(time.time() - t0) / 60:.1f} min")

val_proba = model.predict_proba(X_val)[:, 1]
holdout_auc = roc_auc_score(y_val, val_proba)
print(f"Scheme A holdout ROC-AUC: {holdout_auc:.4f}  (expected ≈ 0.861 ± 0.003)")

## 5. Persist holdout predictions (DeLong artifact)

EXP-001 will run `delong_roc_test` against these scores on the same rows.

In [ ]:
pd.DataFrame(
    {"TransactionID": val_ids, "y_true": y_val.to_numpy(), "score": val_proba}
).to_csv("holdout_pred_exp000.csv", index=False)
print("Saved holdout_pred_exp000.csv")

## 6. Score the competition test set

Known schema quirk: `test_identity.csv` uses dashes in column names (`id-01` … `id-38`) while
the training identity table uses underscores (`id_01`). Rename before applying the feature
contract, then `reindex` to the training feature list (any column absent in test becomes NaN →
filled with 0, matching the production imputation).

In [ ]:
test_transaction = pd.read_csv(DATA_DIR / "test_transaction.csv")
test_identity = pd.read_csv(DATA_DIR / "test_identity.csv")
test_identity.columns = [c.replace("id-", "id_") for c in test_identity.columns]

df_test = test_transaction.merge(test_identity, on="TransactionID", how="left")
del test_transaction, test_identity

X_test = df_test.reindex(columns=feature_list).fillna(0.0)
assert list(X_test.columns) == feature_list, "feature contract violated"

test_proba = model.predict_proba(X_test)[:, 1]
submission = pd.DataFrame(
    {"TransactionID": df_test["TransactionID"], "isFraud": test_proba}
)
submission.to_csv("submission.csv", index=False)
print(submission.head())
print(f"Rows: {len(submission):,} (expected 506,691)")

## 7. Before submitting (author checklist)

1. Holdout AUC within 0.861 ± 0.003 — if not, the reproduction failed: investigate, do NOT submit.
2. `docs/kaggle/submission-log.md`: SUB-001 row completed (commit hash, holdout AUC) BEFORE upload.
3. Submit via notebook UI (Submit to competition) or:
   `kaggle competitions submit ieee-fraud-detection -f submission.csv -m "EXP-000 baseline: production pipeline reproduction"`
4. Fill public/private LB back into `submission-log.md` and `experiment-registry.md` (EXP-000).